# Generación de Texto con modelos GPT

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion5/1-text-generation.ipynb)

Autor: Álvaro José Cabrera Lozano, Claudia Lorena Áragon, Josué Cobaleda
Curso: Procesamiento de Lenguaje Natural  
Universidad Icesi

**INTRODUCCIÓN**

En este proyecto se exploró el uso de modelos generativos tipo GPT para la generación automática de texto en español.
Inicialmente, se trabajó con un modelo pre-entrenado (GPT-2 en español) para analizar su comportamiento en tareas de generación de texto a partir de un prompt.
Posteriormente, se realizó un proceso de fine-tuning utilizando un dataset alternativo basado en noticias, con el objetivo de adaptar el modelo a un dominio más informativo y técnico.
A lo largo del proyecto, se evaluó el impacto del contexto inicial, los parámetros de generación y el entrenamiento del modelo en la calidad del texto generado.

**DESCRIPCIÓN DEL DATASET**

En este proyecto se utilizó el dataset CC News, disponible en HuggingFace, el cual contiene artículos de noticias de diferentes dominios.
Se realizó un filtrado para seleccionar textos relacionados con ciberseguridad, utilizando palabras clave como "hack", "cyber" y "security".
Este dataset permite evaluar el comportamiento del modelo GPT en un contexto técnico, a diferencia del dataset original de chistes.

## Refenencia

- **Dataset:** https://huggingface.co/datasets/cc_news  

- **Paper:** https://arxiv.org/abs/1706.03762  

- **Libro:** https://www.oreilly.com/library/view/natural-language-processing/9781098136789/  

- **Modelo:** https://huggingface.co/datificate/gpt2-small-spanish  

- **Tutorial:** https://huggingface.co/blog/how-to-train  

In [ ]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

## Generative pre-training Transformer - GPT

![](https://github.com/Ohtar10/icesi-nlp/blob/main/assets/gpt.png?raw=1)

Los modelos tipo GPT, introducidos por Radfor, et.al., de OpenAI, al igual que los modelos BERT, hacen uso extensivo de la arquitectura de transformers como hemos estado viendo. Las diferencias claves se podrían resumir en:

1. GPT utiliza bloques de **Transformer Decoder** encadenados, mientras que el modelo BERT utiliza bloques de *Transformer Encoder*
2. GPT se centra en la generación de texto basado en un contexto, la tarea principal es la predicción del siguiente token en la secuencia, mientras que BERT se centra en el completado de partes de una secuencia, en función de un contexto anterior y posterior a la secuencia de entrada. Entonces BERT se centra en la construicción de representación de lenguage, mientras que GPT se centra en la generación de texto en función de un contexto.

Sin embargo, ambos se basan en la misma premisa de pre-entrenar el modelo en tareas no-supervisadas o semi-supervisadas para que el modelo aprenda las representaciones semánticas del lenguage y luego al modelo se le pueda hacer fine tuning a tareas posteriores.

In [ ]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer


device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "mrm8488/spanish-gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

model.config.pad_token_id = tokenizer.pad_token_id

model.resize_token_embeddings(len(tokenizer))

model

In [ ]:
modules = [m for m, _ in model.named_modules()]
modules

Este bloque permite analizar las probabilidades que el modelo asigna a los posibles siguientes tokens a partir de un texto inicial.
Se observa que el modelo predice palabras con mayor probabilidad basándose en patrones aprendidos durante el entrenamiento.
Esto evidencia que GPT no genera texto de manera aleatoria, sino que selecciona palabras según distribuciones de probabilidad.

In [ ]:
text = "Un hacker logró acceder a"
best = 10

with torch.no_grad():
    tokens = tokenizer(text, return_tensors='pt')['input_ids'].to(device)
    print("Dimensiones de la entrada:", tokens.shape)
    output = model(input_ids=tokens)
    print("Dimensiones de la salida:", output.logits.shape)
    output = output.logits[0, -1, :]
    print("Dimensiones del último token de la secuencia:", output.shape)
    probs = torch.softmax(output, dim=-1)
    print("Dimensiones de la probabilidad de los tokens:", probs.shape)
    sorted_probs = torch.argsort(probs, dim=-1, descending=True)
    print({tokenizer.decode(token): f"{prob.cpu().numpy() * 100:.2f}%" for token, prob in zip(sorted_probs[:best], probs[sorted_probs[:best]])})

Al modificar el prompt hacia un contexto de ciberseguridad, se observa que el modelo adapta sus predicciones de manera coherente.
En este caso, asigna mayor probabilidad a palabras como "la", "los", "un" y "una", lo cual sugiere que espera continuar la frase con sustantivos relacionados, como "la red", "los sistemas" o "un servidor".
Esto evidencia que el modelo logra capturar patrones lingüísticos adecuados al contexto proporcionado.

Sin embargo, el modelo no tiene conocimiento real sobre ciberseguridad, sino que basa sus predicciones en patrones aprendidos durante el entrenamiento, lo que puede llevar a generar texto plausible pero no necesariamente preciso.

## Implementando una función de generación

Ahora, la idea es que este modelo nos sirva para generar texto de forma recurrente e incremental. En la última capa de los modelos tipo GPT encontrarémos un tensor con forma $(b, s, v)$, donde:

- $b$: Es el tamaño del bache, o la cantidad de secuencias a procesar.
- $s$: Es la longitud de la secuencia de entrada.
- $v$: Es el tamaño del vocabulario del modelo, cuantos tokens soporta.

Pero este es el tensor de salida, por qué tiene la forma de la secuencia de entrada?, porque cada posición en la salida corresponde a la la predicción del siguiente token de esa posición en la secuencia de entrada. En otras palabras, lo que obtenemos como predicción, es una secuencia de igual tamaño a la de entrada, movida un token hacia adelante, lo que efectivamente nos predice un solo token a la vez y por ende, el token que nos insteresa, es el último.

Lo que obtenemos en este tensor es además los logits de TODO el vocabulario del modelo, con los cuales podemos calcular las probabilidades de que cada uno sea el que continue en la secuencia. Hay varias formas de decodificar el siguiente token, la más fácil de implementar sería una decodificación codiciosa (greedy) del siguiente token, que consiste simplemente en seleccionar el token con la probabilidad más alta. Este es un enfoque simple y efectivo para algunos casos, pero al mismo tiempo sufre de poca variabilidad e incluso puede caer en generación repetitiva.

Otra opción es el muestreo, ya que justamente podemos obtener probabilidades del siguiente token, lo más lógico sería muestrear con esas opciones de probabilidad, de este modo podemos obtener mayor diversidad a la hora de generar el texto, al costo eso si de que haya mayor aleatoridad ya que se le daría la oportunidad a incluso tokens con baja probabilidad, de ser seleccionados.

Otra opción podría ser un balanceo entre una decodificación greedy y una por muestreo, en función de otro hiperparámetro que podemos definir. Esta sería una técnica muy común en el contexto de Reinforcement Learning llamade e-greedy. Se hace la aclaración de que en este ejemplo no harémos nada de RL, solamente se hace mención de esta técnica para balancear entre explotación y exploración.

In [ ]:
import torch.nn as nn
import numpy as np
import pandas as pd
from typing import Optional, Tuple
from transformers.tokenization_utils_base import PreTrainedTokenizerBase


def generate(
        model: nn.Module,
        tokenizer: PreTrainedTokenizerBase,
        start: str,
        max_length: int = 100,
        eps: float = 0.5,
        top_n: int = 5,
        return_iterations: bool = False,
        device: str = "cpu") -> Tuple[str, Optional[pd.DataFrame]]:

    output = [start]
    iterations = []
    with torch.no_grad():
        input_ids = tokenizer(output[-1], return_tensors='pt')['input_ids'].to(device)
        for _ in range(max_length):
            # Tomamos los logits producidos por la última capa del modelo
            # Estos corresponden al siguiente token por cada posición de la cadena
            logits = model(input_ids=input_ids).logits
            # Por lo tanto, el que nos interesa es el último, que correspondería a la
            # predicción del siguiente token después del final de la cadena original
            # A este aplicamos un softmax para obtener las probabilidades por cada
            # token del vocabulario para estar presente en la cadena.
            probs = torch.softmax(logits[0, -1, :], dim=-1)
            # Simplemente ordenamos por probabilidad de forma descendente
            sorted_tokens = torch.argsort(probs, dim=-1, descending=True)

            # Utilizamos una politica tipo e-greedy para obtener el siguiente token de la secuencia
            # Un eps>=1 quiere decir que siempre se va seleccionar el token de forma 'greedy', es decir
            # siempre se toma el token con probabilidad más alta.

            # Un eps=0 quiere decir que siempre se va a muestrear el siguiente token en función
            # de las probabilidades de cada token

            # Un 0<eps<1 va a balancear de forma binomial entre tomar el token con la
            # probabilidad más alta y muestrear el token en función de sus probabilidades.
            if np.random.random_sample(1)[0] < eps:
                # Se toma el mejor token
                next_token = sorted_tokens[0].unsqueeze(dim=0)
            else:
                # Se muetrea el token de la probabilidad de distribución
                next_token = torch.multinomial(probs, 1)

            if return_iterations:
                # Mantenemos pista de todas las iteraciones para análisis
                iteration = {'input': ''.join(output)}
                best_n = sorted_tokens[:top_n].cpu().tolist()
                choices = {f'Choice #{choice+1}': f'{tokenizer.decode(token)} ({prob:.4f})' for choice, (token, prob) in enumerate(zip(best_n, probs[best_n].cpu().tolist()))}
                iteration.update(choices)
                iterations.append(iteration)

            output.append(tokenizer.decode(next_token))
            input_ids = torch.cat([input_ids, next_token.unsqueeze(dim=0)], dim=-1)

        output_text = ''.join(output)
        if not return_iterations:
            return output_text, None
        else:
            df = pd.DataFrame(iterations)
            return output_text, df

Ahora observemos que pasa cuando generamos texto con nuestra función y algunos parámetros.

Primero, observemos que pasa cuando pasamos un `eps=1` que quiere decir que la generación va a ser de tipo greedy:

In [ ]:
output_text, iterations_df = generate(model, tokenizer, text, max_length=15, eps=1.0, top_n=10, return_iterations=True, device=device)
print(output_text)
iterations_df.head(15)

##Interpretaciòn
Al analizar las probabilidades de los tokens, se evidencia que el modelo asigna alta probabilidad a palabras coherentes con el contexto, como "base", "datos", "compañía" y "seguridad".
En algunos casos, se observan probabilidades extremadamente altas (cercanas a 1), lo que indica que el modelo tiene alta certeza en ciertas secuencias comunes, como "base de datos".

Se observa que el modelo genera texto coherente y contextualizado a partir del prompt relacionado con ciberseguridad.
En particular, la secuencia generada "Un hacker logró acceder a la base de datos de la compañía de seguros" muestra una estructura lógica y realista dentro del dominio tecnológico.
Esto indica que el modelo ha aprendido patrones lingüísticos asociados a contextos de acceso a sistemas y manejo de información.

Observamos como el input progresa a la vez que las opciones de tokens que hay. Sin importar cuantas veces invoquemos a la función con los mismos parámetros, siempre vamos a obtener los mismos resultados.

Ahora, observemos que pasa si introducimos exploración al reducir el `eps=0.5`, lo cual nos dice que aproximadamente la mitad de las veces va a elegir el siguiente token muestreando y la otra mitad explotando.

In [ ]:
output_text, _ = generate(model, tokenizer, text, max_length=100, eps=0.5, device=device)
print(output_text)

Al utilizar un valor menor de eps=0.5, se introduce mayor variabilidad en la selección de tokens, lo que incrementa la creatividad del texto generado, pero también reduce su coherencia.

#Analisis
Al aumentar la longitud de generación del texto, se observa que el modelo mantiene coherencia en las primeras frases, construyendo una idea lógica relacionada con el acceso a información en un contexto de ciberseguridad.
Sin embargo, a medida que el texto se extiende, el modelo comienza a perder consistencia semántica, introduciendo términos que no están directamente relacionados con el contexto inicial, como referencias a “Astronomía” o repeticiones innecesarias.

* Este comportamiento evidencia que el modelo genera texto de manera secuencial y local, sin una comprensión global del discurso, lo que provoca acumulación de errores a medida que aumenta la longitud del texto.2*

* Esto demuestra una de las principales limitaciones de los modelos generativos como GPT: aunque pueden generar texto fluido, no garantizan consistencia global en secuencias largas.

En este caso, cada vez que invoquemos a la función, vamos a obtener una respuesta diferente, a veces más coherente y otras veces menos. Vale la pena realizar varias pruebas para observar los resultados hasta encontrar un balance.

### Generando texto con las utilidades del modelo

Ahora, la clase de Huggingface implementa la función `generate` que hace la labor de generación por nosotros, incluyendo las opciones de muestreo y explotación como hemos observado. Solo que además permite otra serie de parámetros y opciones para controlar la generación de texto. Para más información se recomienda estudiar:

- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246), Capitulo 5
- https://huggingface.co/docs/transformers/v4.41.3/en/main_classes/text_generation#transformers.GenerationConfig
- https://huggingface.co/docs/transformers/v4.41.3/en/main_classes/text_generation#transformers.GenerationMixin.generate

In [ ]:
output = model.generate(tokens, pad_token_id=tokenizer.eos_token_id, max_length=100, do_sample=True, temperature=0.5, top_k=0)
print(tokenizer.decode(output[0]))

### Análisis con parámetros originales

Al utilizar los parámetros originales del modelo (top_k=0), se observa que la generación de texto inicia de manera coherente, describiendo un escenario relacionado con ciberseguridad.
Sin embargo, a medida que avanza la secuencia, el modelo comienza a generar preguntas repetitivas como “¿Qué?” y “¿Para qué?”, perdiendo coherencia y estructura en el discurso.

##### Nota
Este resultado demuestra que la generación de texto requiere un ajuste cuidadoso de los parámetros para evitar problemas como la repetición o pérdida de coherencia.

In [ ]:
output = model.generate(tokens, pad_token_id=tokenizer.eos_token_id, max_length=100, do_sample=True, temperature=0.5, top_k=50)
print(tokenizer.decode(output[0]))

Al modificar el parámetro (top_k a 50), se observa una mejora significativa en la calidad del texto generado.
El modelo logra construir una secuencia coherente y estructurada, manteniendo el contexto de ciberseguridad sin caer en repeticiones excesivas.
En este caso, el texto generado describe de manera lógica un acceso a un sistema de seguridad, lo cual es consistente con el prompt inicial.

## Comparación de resultados

Se realizaron dos experimentos: uno con los parámetros originales del modelo y otro modificando el parámetro top_k.
En el primer caso, el modelo presentó problemas de repetición y pérdida de coherencia.
En el segundo caso, al limitar el número de tokens candidatos (top_k=50), se obtuvo una mejora significativa en la calidad del texto generado, reduciendo la repetición y manteniendo mejor el contexto.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("cc_news", split="train[:1%]")
dataset

In [ ]:
dataset[0]

In [ ]:
dataset.set_format('pandas')
df = dataset.to_pandas()
df.head(10)

# Nota :
Se utilizó la columna text del dataset CC News, la cual contiene el contenido principal de los artículos.
Se realizó una limpieza básica eliminando valores no válidos y limitando la cantidad de datos para facilitar el entrenamiento.

In [ ]:
df['Palabras por texto'] = df['text'].str.split().apply(len)
df['Palabras por texto'].median()

Aquí podemos observar que la mediana de longitud en terminos de palabras es de 31. Esto es esperado, pues los chistes deben ser cortos por naturaleza. Por otra parte, es bastante claro que el corpus original del modelo pre-entrenado contenía texto muy diferente a este, por lo que la calidad de los resultados, sin hacer mayores modificaciones puede que no sea buena.

Sin embargo, a manera demostrativa, continuarémos con el ejercicio, prepararémos el conjunto de datos para entrenamiento.
# Analisis descrptivo:
Se calculó la cantidad de palabras por texto en el dataset de noticias, obteniendo una mediana de aproximadamente 215 palabras por documento.
Esto indica que los textos son considerablemente más largos en comparación con el dataset original de chistes, el cual contenía textos más cortos y simples.

In [ ]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        return tokenizer(
            examples['text'],
            max_length=max_len,
            truncation=True,
            padding="max_length"
        )
    return _preprocess_function

In [ ]:
def preprocess_function(max_len):
    def _preprocess_function(examples):
        return tokenizer(examples['text'], max_length=max_len, truncation=True)
    return _preprocess_function


Los modelos GPT no esperan otra cosa más que los `input_ids`, por lo que retirarémos todas las demás columnas del dataset ya que no nos son de utilidad en este momento.

In [ ]:
dataset.reset_format()

tokenized_dataset = dataset.map(
    preprocess_function(max_len=64),
    batched=True
)

tokenized_dataset = tokenized_dataset.remove_columns(
    [col for col in tokenized_dataset.column_names if col not in ['input_ids', 'attention_mask']]
)

tokenized_dataset = tokenized_dataset.train_test_split(train_size=0.9)

tokenized_dataset.set_format('torch')

tokenized_dataset

Se aplicó la función de preprocesamiento al dataset completo, convirtiendo los textos en secuencias de tokens mediante el tokenizer del modelo.
Posteriormente, se eliminaron las columnas innecesarias, conservando únicamente input_ids y attention_mask.
Finalmente, el dataset fue dividido en conjuntos de entrenamiento (90%) y prueba (10%), y convertido al formato de tensores de PyTorch para su uso en el entrenamiento del modelo.

*Se observa que el dataset resultante contiene más de 6000 ejemplos para entrenamiento, lo cual proporciona suficiente información para ajustar el modelo a un nuevo dominio, en este caso, textos informativos.

In [ ]:
from transformers import DataCollatorForLanguageModeling
from transformers import Trainer, TrainingArguments

batch_size = 4 if IN_COLAB else 2
logging_steps = len(tokenized_dataset['train']) // batch_size
# Definimos los parámetros globales de entrenamiento
training_args = TrainingArguments(
    output_dir='./hf-gpt',
    num_train_epochs=10,
    learning_rate=2e-5,
    per_device_eval_batch_size=batch_size,
    per_device_train_batch_size=batch_size,
    weight_decay=0.01,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    disable_tqdm=False,
    logging_steps=logging_steps,
    report_to='none'
)

# Y definimos el entrenador, especificando el modelo, datasets y el tokenizador
trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['test'],
)

Se realizó el entrenamiento del modelo utilizando el dataset de noticias, permitiendo adaptar el modelo GPT-2 a un dominio más informativo.
Durante el entrenamiento se observó la optimización de la función de pérdida, lo que indica que el modelo está aprendiendo patrones del nuevo dataset.

In [ ]:
%%time
trainer.train()

#Resultados del entrenamiento

Se llevó a cabo el proceso de fine-tuning del modelo GPT-2 utilizando el dataset de noticias.
Durante el entrenamiento, se observó una disminución progresiva de la función de pérdida, lo que indica que el modelo logró aprender patrones del nuevo dominio.
El valor final de la pérdida sugiere una adecuada adaptación del modelo, considerando que se utilizó un número limitado de épocas.
* Este proceso evidencia que los modelos pre-entrenados pueden ser adaptados a nuevos contextos mediante fine-tuning, mejorando su capacidad de generar texto en dominios específicos.
* Sin embargo, debido a las restricciones computacionales y al número reducido de épocas, el modelo no alcanza un ajuste completo, lo cual podría mejorarse con mayor tiempo de entrenamiento.



In [ ]:
output = model.generate(
    tokens,
    pad_token_id=tokenizer.eos_token_id,
    max_length=100,
    do_sample=True,
    temperature=0.7,
    top_k=50,
    top_p=0.9
)

print(tokenizer.decode(output[0]))

#Evaluación del modelo entrenado

Tras el proceso de fine-tuning, se observa que el modelo genera texto más coherente y estructurado en comparación con el modelo original.
En este caso, la secuencia generada mantiene una narrativa lógica dentro del contexto de ciberseguridad, incluyendo elementos como acceso a información y contenido digital.
Además, el modelo presenta una mejor fluidez y menor repetición de palabras, lo que indica una mejora en la calidad de la generación.



In [ ]:
output_text, _ = generate(
    model,
    tokenizer,
    text,
    max_length=100,
    eps=0.7,
    device=device
)

print(output_text)

El modelo logra generar una secuencia coherente al inicio, manteniendo el contexto de ciberseguridad y produciendo una narrativa lógica.
Sin embargo, a medida que avanza la generación, se observa repetición de frases como “y, por supuesto, no fue detectado”, lo que indica una pérdida de diversidad en el texto generado.

# RESULTADOS

Durante el entrenamiento, se observó una disminución progresiva de la función de pérdida, lo que indica que el modelo logró aprender patrones del dataset de noticias.
Al evaluar el modelo entrenado, se evidenció una mejora significativa en la coherencia y estructura del texto generado, especialmente en contextos relacionados con ciberseguridad.
Sin embargo, también se identificaron limitaciones, como la repetición de frases en secuencias largas, lo cual es característico de los modelos generativos.

## Conclusiones
- Los modelos BERT y GPT son muy similares, aunque ambos tienen diferencias claras en cuanto a su estructura y manera de entrenamiento.
- Sin embargo, ambos pueden utilizarse para el mismo tipo de tareas posteriores, lo cual sustenta la importancia del pre-entrenamiento y la construcción de embeddings de buena calidad.
- En los modelos generativos, tiende a ver un dilema de tipo exploración-explitación, al explotar los resultados, podemos ser más precisos, per al mismo tiempo más monótonos, mientras que explorando podemos ser más creativos y diversos, pero al mismo tiempo terminar con texto incoherente, difuso o alucinante. Es necesario evaluar la tarea a la mano para escoger el ajuste adecuado entre estas dos técnicas de decodificación.
- Los modelos generativos de texto no son más que una gran probabilidad de distribución y esta a su vez es completamente dependiente de los datos con los que fue entrenada. Es aquí donde se hace sumamente importante obtener y curar los conjuntos de datos con los que se entrena, de lo contrario se puede terminar con un modelo de mala calidad para la tarea en especifico.
- Diferentes estrategias de decodificación entregan resultados diferentes, vale la pena hacer una exploración de los resultados y ajustar los hiperparámetros para obtener los resultados deseados según el objetivo.

* Este proyecto permitió evidenciar la capacidad de los modelos GPT para generar texto coherente a partir de un contexto inicial, así como su sensibilidad a los parámetros de generación.
El proceso de fine-tuning demostró ser una técnica efectiva para adaptar modelos pre-entrenados a dominios específicos, mejorando la calidad del texto generado.
No obstante, los modelos generativos presentan limitaciones en la consistencia global y pueden producir repeticiones o información no verificada.
En general, el proyecto destaca la importancia del diseño del prompt, la selección del dataset y el ajuste de hiperparámetros para obtener resultados óptimos en tareas de generación de texto.
* Como aporte adicional, se realizaron experimentos comparando diferentes configuraciones de generación, así como el comportamiento del modelo antes y después del fine-tuning.
Esto permitió identificar cómo los parámetros influyen en la calidad del texto y cómo el entrenamiento mejora la adaptación a nuevos contextos.
Asimismo, se evidenció la importancia de utilizar datasets acordes al dominio de interés para obtener resultados más relevantes.